# R3-S1 - corruption suite: validation-sample run (Colab)

Applies the four frozen corruptions (`mp3_64`, `mp3_128`, `noise_20`, `noise_10`) to a small seeded sample of Protocol B **selection**-role
recordings, to check the builder on real audio. CPU runtime is enough. Run all cells top to bottom; it is safe to rerun after a disconnect.

**Firewall.** Only selection-role validation recordings are read (genuine, MeloTTS, VITS, XTTS). Test, calibration and operating recordings and
ChatterboxTTS are never listed or read. No model scores anything and **no audio is written**. Building the stored corrupted copies of the locked test
recordings is R5-S1, after GR4.

**Produces** in `reports/r3/` on Drive: `r3_s1_corruption_sample.json`, `r3_s1_corruption_sample_variants.csv`, and the evidence report
`r3_s1_colab_<timestamp>.json`. It also reports the ffmpeg build, because MP3 bytes can differ between ffmpeg builds.

In [1]:
import os, sys, json, subprocess, time, hashlib, platform, shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get("ECHOPROOF_ROOT", "/content/drive/MyDrive/PAI&AIES Project"))
assert ROOT.exists(), f"project folder not found: {ROOT}"

SOURCES = {"corruption.py": "src/echoproof/data/corruption.py", "corruption_sample.py": "src/echoproof/data/corruption_sample.py",
           "measure.py": "src/echoproof/eda/measure.py", "selection.py": "src/echoproof/eda/selection.py", "eda_init.py": "src/echoproof/eda/__init__.py",
           "r2_s1.py": "src/echoproof/features/r2_s1.py", "stats.py": "src/echoproof/features/stats.py", "acoustic.py": "src/echoproof/features/acoustic.py",
           "cohort.py": "src/echoproof/data/cohort.py", "splits.py": "src/echoproof/data/splits.py", "chunks.py": "src/echoproof/data/chunks.py",
           "contracts.py": "src/echoproof/contracts.py", "artifacts.py": "src/echoproof/artifacts.py", "seeding.py": "src/echoproof/training/seeding.py",
           "r3_s1_corruption_sample.py": "scripts/r3_s1_corruption_sample.py", "test_r3_s1_corruption.py": "tests/test_r3_s1_corruption.py",
           "test_r3_s1_sample.py": "tests/test_r3_s1_sample.py", "test_r2_s1_features.py": "tests/test_r2_s1_features.py"}
missing = [k for k, rel in SOURCES.items() if not (ROOT / rel).exists()]
assert not missing, f"not found under {ROOT}: {missing} - sync these files to Drive and rerun"
SHA = {k: hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() for k, rel in SOURCES.items()}
print(json.dumps({k: v[:12] for k, v in SHA.items()}, indent=2))

Mounted at /content/drive
{
  "corruption.py": "4541e98f019e",
  "corruption_sample.py": "dbc71b647604",
  "measure.py": "e9e257f3d8c6",
  "selection.py": "c513a1d27bb1",
  "eda_init.py": "afca4d98cfc7",
  "r2_s1.py": "06346d7b1bc1",
  "stats.py": "ae649f3cface",
  "acoustic.py": "2961146cc395",
  "cohort.py": "39a8b0091424",
  "splits.py": "2ed05ad91116",
  "chunks.py": "ef3fdc6b4790",
  "contracts.py": "20483ebcff37",
  "artifacts.py": "5e38663230cf",
  "seeding.py": "4f85c4c23142",
  "r3_s1_corruption_sample.py": "8fdb648e5f39",
  "test_r3_s1_corruption.py": "12f8954815ca",
  "test_r3_s1_sample.py": "f74ba82af284",
  "test_r2_s1_features.py": "9941b2ca93c6"
}


## 0. ffmpeg with the MP3 encoder

In [2]:
def ffmpeg_ready():
    exe = shutil.which("ffmpeg")
    if not exe:
        return None
    enc = subprocess.run([exe, "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
    return exe if "libmp3lame" in enc else None

FFMPEG = ffmpeg_ready()
if FFMPEG is None:                                    # Colab normally ships ffmpeg; install only if it is missing
    print(subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], capture_output=True, text=True).stdout[-500:])
    FFMPEG = ffmpeg_ready()
assert FFMPEG, "ffmpeg with libmp3lame is not available on this runtime"
FFMPEG_VERSION = subprocess.run([FFMPEG, "-version"], capture_output=True, text=True).stdout.splitlines()[0]
print(FFMPEG, "|", FFMPEG_VERSION)

/usr/bin/ffmpeg | ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [3]:
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["ECHOPROOF_FFMPEG"] = FFMPEG
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "src"))

import numpy, scipy, librosa, pyarrow, pandas
ENV = {"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "librosa": librosa.__version__,
       "pyarrow": pyarrow.__version__, "pandas": pandas.__version__, "platform": platform.platform(), "cpu_count": os.cpu_count(),
       "ffmpeg": FFMPEG_VERSION}
print(json.dumps(ENV, indent=2))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

INPUTS = {"cohort": ROOT / "data/manifests/cohort_v1.parquet", "split": ROOT / "data/splits/protocol_B_v1.parquet",
          "chunks": ROOT / "data/processed/chunks_v1.parquet", "split_report": ROOT / "reports/r1/split_report.json"}
for name, path in INPUTS.items():
    assert path.exists(), f"missing locked input {name}: {path}"
EXPECTED_COHORT_FILE_SHA256 = "51af27a2dd5c0577bf9c232287ac73b2f66af472ce3308721af83b2e48e8e816"
INPUT_SHA = {name: sha256_file(path) for name, path in INPUTS.items() if name != "split_report"}
print({k: v[:12] for k, v in INPUT_SHA.items()})
assert INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256, "the cohort manifest on Drive is not the frozen GR1 file - stop and compare it with the local copy"

{
  "python": "3.13.18",
  "numpy": "2.4.6",
  "scipy": "1.16.3",
  "librosa": "0.11.0",
  "pyarrow": "23.0.1",
  "pandas": "2.2.3",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "cpu_count": 2,
  "ffmpeg": "ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers"
}
{'cohort': '51af27a2dd5c', 'split': 'd910f5b2798e', 'chunks': 'd3f8b8d6d305'}


## 1. Unit tests on this runtime

In [4]:
proc = subprocess.run([sys.executable, "-m", "pytest", "tests/test_r3_s1_corruption.py", "tests/test_r3_s1_sample.py", "-q", "-p", "no:cacheprovider"],
                      cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(proc.stdout[-3000:] + proc.stderr[-1500:])
UNIT_TESTS_PASSED = proc.returncode == 0
PYTEST_SUMMARY = next((l.strip() for l in reversed(proc.stdout.splitlines()) if " passed" in l or " failed" in l or " error" in l), "no summary line")
print("unit tests:", "PASSED" if UNIT_TESTS_PASSED else "FAILED", "|", PYTEST_SUMMARY)
assert UNIT_TESTS_PASSED, "do not build on a runtime where the tests fail"

.....................................                                    [100%]
37 passed in 69.79s (0:01:09)

unit tests: PASSED | 37 passed in 69.79s (0:01:09)


## 2. Plan the sample and copy only the shards it needs

In [5]:
from echoproof.data import corruption, corruption_sample as cs

PER_COMPONENT, SEED = 8, 42
LOADED = cs.load_inputs(ROOT, SEED)                      # frozen-hash checks, test dropped, Chatterbox rejected, no audio read
SAMPLE = cs.draw_sample(LOADED["eda"].items, PER_COMPONENT, SEED)
needed = sorted({i["shard"] for i in SAMPLE})
SHARD_SRC, SHARD_DIR = ROOT / "data/processed/cohort_v1", Path("/content/shards")
SHARD_DIR.mkdir(parents=True, exist_ok=True)
print(len(SAMPLE), "recordings of", len(LOADED["eda"].items), "selection recordings; shards:", needed)
t0 = time.perf_counter()
for name in needed:
    src, dst = SHARD_SRC / name, SHARD_DIR / name
    assert src.exists(), f"shard missing on Drive: {src}"
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        continue
    for attempt in range(4):
        try:
            tmp = dst.with_name(dst.name + ".part")
            shutil.copyfile(src, tmp)
            assert tmp.stat().st_size == src.stat().st_size, "size differs after copy"
            os.replace(tmp, dst)
            break
        except OSError as exc:
            print(f"  {name}: attempt {attempt + 1} failed ({exc}); retrying")
            time.sleep(5 * 2 ** attempt)
    else:
        raise RuntimeError(f"could not copy {name}")
COPY_SECONDS = round(time.perf_counter() - t0, 1)
print("shards ready |", COPY_SECONDS, "s")

32 recordings of 2000 selection recordings; shards: ['shard_00.tar', 'shard_01.tar', 'shard_02.tar', 'shard_03.tar', 'shard_04.tar', 'shard_07.tar', 'shard_08.tar', 'shard_09.tar', 'shard_10.tar', 'shard_12.tar']
shards ready | 59.4 s


## 3. Run the sample

In [6]:
OUT_DIR = ROOT / "reports/r3"
if (OUT_DIR / cs.REPORT_NAME).exists():
    print("already built - loading the existing report")
    REPORT = json.loads((OUT_DIR / cs.REPORT_NAME).read_text())
else:
    REPORT = cs.run(ROOT, SHARD_DIR, OUT_DIR, per_component=PER_COMPONENT, seed=SEED, ffmpeg=FFMPEG, progress=print)
print(json.dumps({k: REPORT[k] for k in ("passed", "checks", "failures", "sample", "seconds_total")}, indent=2))
print(json.dumps(REPORT["conditions"], indent=2))

plan adfca8c9faf0: sampling 32 of 2000 selection recordings, 4 conditions each
phonemedf:genuine:1098-133695-0008: done
phonemedf:xtts:1098-133695-0058: done
phonemedf:genuine:1963-142393-0000: done
phonemedf:melo:1553-140048-0001: done
phonemedf:melo:1578-140049-0027: done
phonemedf:melo:1898-145724-0037: done
phonemedf:vits:19-198-0006: done
phonemedf:xtts:19-198-0030: done
phonemedf:xtts:1963-142393-0050: done
phonemedf:genuine:2007-149877-0020: done
phonemedf:melo:302-123504-0000: done
phonemedf:genuine:322-124146-0018: done
phonemedf:melo:3112-9555-0073: done
phonemedf:vits:322-124146-0013: done
phonemedf:vits:3235-11599-0021: done
phonemedf:xtts:3235-11599-0024: done
phonemedf:genuine:4813-248641-0042: done
phonemedf:genuine:5192-19396-0026: done
phonemedf:melo:4853-27670-0036: done
phonemedf:vits:5192-19397-0001: done
phonemedf:melo:6019-3185-0021: done
phonemedf:vits:5393-19218-0001: done
phonemedf:vits:5393-19219-0059: done
phonemedf:xtts:5393-19218-0056: done
phonemedf:xtts:5

## 4. Independent cross-checks

In [7]:
import csv

problems = []
TABLE_PATH = OUT_DIR / cs.TABLE_NAME
if sha256_file(TABLE_PATH) != REPORT["outputs"][cs.TABLE_NAME]["sha256"]:
    problems.append("variant table hash differs from the report")
with open(TABLE_PATH, newline="") as handle:
    ROWS = list(csv.DictReader(handle))
if len(ROWS) != len(SAMPLE) * 4 or REPORT["sample"]["variants_built"] != len(ROWS):
    problems.append(f"{len(ROWS)} rows for {len(SAMPLE)} recordings x 4 conditions")
import pyarrow.parquet as pq
SPLIT_ROWS = [json.loads(r["payload_json"]) for r in pq.read_table(INPUTS["split"]).to_pylist()]
selection_ids = {r["recording_id"] for r in SPLIT_ROWS if r["partition"] == "validation" and r["validation_role"] == "selection"}
parents = {r["parent_id"] for r in ROWS}
if not parents <= selection_ids:
    problems.append("a parent recording is not a selection-role recording")
if any(":chatterbox:" in p for p in parents):
    problems.append("Chatterbox recordings appear")
if {r["component"] for r in ROWS} != {"genuine", "melo", "vits", "xtts"}:
    problems.append("components differ from genuine/melo/vits/xtts")
if sorted({r["condition"] for r in ROWS}) != sorted(corruption.CONDITION_NAMES):
    problems.append("conditions differ from the frozen four")
if REPORT["spec"]["spec_sha256"] != corruption.SPEC_SHA256:
    problems.append("report spec hash differs from the code")

# Recompute one variant per condition from the shards, independently of the run loop, and compare PCM hashes with the table.
import tarfile
from echoproof.features.r2_s1 import _read_member
by_id = {i["recording_id"]: i for i in SAMPLE}
RECOMPUTED = []
for condition in corruption.CONDITION_NAMES:
    row = next(r for r in ROWS if r["condition"] == condition)
    item = by_id[row["parent_id"]]
    with tarfile.open(SHARD_DIR / item["shard"]) as tar:
        wave = _read_member(tar, item)
    sha = corruption.corrupt(wave, item["recording_id"], condition, seed=SEED, ffmpeg=FFMPEG).pcm_sha256
    RECOMPUTED.append({"variant_id": row["variant_id"], "match": sha == row["pcm_sha256"]})
    if sha != row["pcm_sha256"]:
        problems.append(f"{row['variant_id']}: recomputed hash differs from the table")
print(RECOMPUTED)

LOCAL_FIXTURE_DIGEST = "2898daefe8ba1f50921e8e92eaad98e9b09545106d73db699c9c28dfb5f86139"      # the same digest on the project machine (ffmpeg 9.0.2 full build); informational only
FIXTURE_MATCHES_LOCAL = REPORT["fixture"]["digest"] == LOCAL_FIXTURE_DIGEST
print("fixture digest equals the project-machine value:", FIXTURE_MATCHES_LOCAL,
      "" if FIXTURE_MATCHES_LOCAL else "(a different ffmpeg build can change MP3 bytes; generate the final R5-S1 copies on one machine and record its build)")
print("problems:", problems or "none")

[{'variant_id': 'phonemedf:genuine:1098-133695-0008@mp3_64', 'match': True}, {'variant_id': 'phonemedf:genuine:1098-133695-0008@mp3_128', 'match': True}, {'variant_id': 'phonemedf:genuine:1098-133695-0008@noise_20', 'match': True}, {'variant_id': 'phonemedf:genuine:1098-133695-0008@noise_10', 'match': True}]
fixture digest equals the project-machine value: True 
problems: none


## 5. Verdict and evidence

In [8]:
checks = {"unit_tests_passed": UNIT_TESTS_PASSED, "run_passed": REPORT["passed"], "run_checks": REPORT["checks"], "run_failures": REPORT["failures"],
          "cohort_manifest_is_frozen_file": INPUT_SHA["cohort"] == EXPECTED_COHORT_FILE_SHA256,
          "split_hash_is_frozen_value": REPORT["inputs"]["split_sha256"] == REPORT["inputs"]["split_sha256_frozen_in_report"],
          "cross_check_problems": problems, "fixture_digest_matches_project_machine": FIXTURE_MATCHES_LOCAL}
verified = (UNIT_TESTS_PASSED and REPORT["passed"] and not REPORT["failures"] and all(REPORT["checks"].values())
            and checks["cohort_manifest_is_frozen_file"] and checks["split_hash_is_frozen_value"] and not problems)
evidence = {"schema_version": 1, "task": "R3-S1", "step": "validation_sample", "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "environment": ENV, "source_sha256": SHA, "input_sha256": INPUT_SHA, "output_sha256": {cs.TABLE_NAME: sha256_file(TABLE_PATH), cs.REPORT_NAME: sha256_file(OUT_DIR / cs.REPORT_NAME)},
            "verdict": {"verified": bool(verified), "pytest_summary": PYTEST_SUMMARY, **checks}, "recomputed": RECOMPUTED, "report": REPORT,
            "timing": {"shard_copy_seconds": COPY_SECONDS}}
path = OUT_DIR / f"r3_s1_colab_{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}.json"
path.write_text(json.dumps(evidence, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(json.dumps(evidence["verdict"], indent=2, default=str))
print("\nevidence written to:", path)

{
  "verified": true,
  "pytest_summary": "37 passed in 69.79s (0:01:09)",
  "unit_tests_passed": true,
  "run_passed": true,
  "run_checks": {
    "all_variants_built": true,
    "repeat_hashes_identical": true,
    "parent_ids_kept": true,
    "only_selection_role": true,
    "no_chatterbox": true,
    "lengths_preserved": true,
    "unique_variant_ids": true
  },
  "run_failures": [],
  "cohort_manifest_is_frozen_file": true,
  "split_hash_is_frozen_value": true,
  "cross_check_problems": [],
  "fixture_digest_matches_project_machine": true
}

evidence written to: /content/drive/MyDrive/PAI&AIES Project/reports/r3/r3_s1_colab_20261110T114146Z.json
